In [ ]:
!pip install rdkit

!pip install torch-geometric
!pip install tensorflow

In [ ]:
!pip install rdkit

In [ ]:
# ----------------------- 1. IMPORTS AND CONSTANTS -----------------------

import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ttest_rel, wilcoxon

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler
from torch_geometric.loader import DataLoader

from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GINConv, global_mean_pool
from torch_geometric.utils import add_self_loops

from rdkit import Chem
from rdkit.Chem import Descriptors

# Sabitler
SEED           = 42
BATCH_SIZE     = 128
LR_BASE        = 1e-4
WEIGHT_DECAY   = 1e-3
EPOCHS         = 60
PATIENCE       = 8
HIDDEN_DIM     = 128
N_LAYERS       = 4
DROPOUT_RATE   = 0.2
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FCL_ALPHA      = 0.25
FCL_GAMMA      = 2.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore", category=UserWarning)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [ ]:
# ----------------------- 2. FEATURIZER VE SMILES→Data -----------------------

class Featurizer:
    def __init__(self, allow_map):
        self.dim, self.map = 0, {}
        for key, vals in allow_map.items():
            sorted_vals = sorted(list(vals))
            self.map[key] = {v: i + self.dim for i, v in enumerate(sorted_vals)}
            self.dim += len(sorted_vals)

    def _encode(self, obj):
        vec = np.zeros(self.dim, dtype=np.float32)
        for method_name, mapping in self.map.items():
            val = getattr(self, method_name)(obj)
            if val in mapping:
                vec[mapping[val]] = 1.0
        return vec

class AtomFeat(Featurizer):
    def n_number(self, atom):            return atom.GetAtomicNum()
    def n_valence(self, atom):           return atom.GetTotalValence()
    def n_hydrogens(self, atom):         return atom.GetTotalNumHs()
    def hybridization(self, atom):       return atom.GetHybridization().name.lower()
    def n_degree(self, atom):            return atom.GetDegree()
    def n_radical_electrons(self, atom): return atom.GetNumRadicalElectrons()
    def is_in_ring(self, atom):          return atom.IsInRing()
    def encode(self, atom):              return self._encode(atom)

class BondFeat(Featurizer):
    def __init__(self, allow_map):
        super().__init__(allow_map)
        self.dim += 1

    def bond_type(self, bond):      return bond.GetBondType().name.lower()
    def conjugated(self, bond):     return bond.GetIsConjugated()

    def encode(self, bond):
        if bond is None:
            vec = np.zeros(self.dim, dtype=np.float32)
            vec[-1] = 1.0
            return vec
        return self._encode(bond)

# Atom and bond feature maps
ATOM_FTR = AtomFeat({
    "n_number":           {1, 5, 6, 8, 9, 11, 15, 16, 17, 20, 35, 53},
    "n_valence":          {0, 1, 2, 3, 4, 5, 6},
    "n_hydrogens":        {0, 1, 2, 3, 4},
    "hybridization":      {"s", "sp", "sp2", "sp3"},
    "n_degree":           {0, 1, 2, 3, 4, 5, 6},
    "n_radical_electrons":{0, 1, 2, 3, 4},
    "is_in_ring":         {True, False}
})
BOND_FTR = BondFeat({
    "bond_type":  {"single", "double", "triple", "aromatic"},
    "conjugated": {True, False}
})

def mol_from_smiles(smiles, randomize=False):
    """
    Converts SMILES → Mol with RDKit. If randomize=True,
    Applies a random rearrangement via MolToSmiles(doRandom=True).
    """
    if randomize:
        base = Chem.MolFromSmiles(smiles)
        smiles = Chem.MolToSmiles(base, doRandom=True)
    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    flag = Chem.SanitizeMol(mol, catchErrors=True)
    if flag != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(mol, Chem.SanitizeFlags.SANITIZE_ALL ^ flag)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    return mol

def pyg_from_mol(mol, y=None):
    """
    Converts an RDKit Mol into a PyG Data object:
    - Feature vector for atoms via ATOM_FTR.encode
    - BOND_FTR.encode for bonds
    """
    x = torch.from_numpy(
        np.stack([ATOM_FTR.encode(atom) for atom in mol.GetAtoms()])
    ).float()

    edge_index_list, edge_attr_list = [], []
    num_atoms = mol.GetNumAtoms()

    # Self-loop ekle (bond=None)
    for i in range(num_atoms):
        edge_index_list.append([i, i])
        edge_attr_list.append(BOND_FTR.encode(None))

    # Real bonds
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        f = BOND_FTR.encode(bond)
        edge_index_list += [[i, j], [j, i]]
        edge_attr_list += [f, f]

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(np.stack(edge_attr_list), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)
    return data

def compute_descriptors(smiles):
    """
    Extracts a 6-dimensional molecule-level descriptor using RDKit:
    MolLogP, MolWt, TPSA, NumHAcceptors, NumHDonors, NumRotatableBonds
    """
    m = Chem.MolFromSmiles(smiles)
    descs = [
        Descriptors.MolLogP(m),
        Descriptors.MolWt(m),
        Descriptors.TPSA(m),
        Descriptors.NumHAcceptors(m),
        Descriptors.NumHDonors(m),
        Descriptors.NumRotatableBonds(m)
    ]
    return np.array(descs, dtype=np.float32)

def add_virtual_node(data):
    """
    Adds a "virtual node" to the end of each graph. This node
    is used only to pass global descriptor information.
    """
    num_nodes = data.x.size(0)
    feat_dim = data.x.size(1)

    # Virtual-node feature: zero vector (global_desc is used inside the model)
    vn_feat = torch.zeros((1, feat_dim), device=data.x.device)
    data.x = torch.cat([data.x, vn_feat], dim=0)

    # Batch tensor'una ekle
    data.batch = torch.cat([data.batch, data.batch.new_full((1,), data.batch[0])], dim=0)

    # Virtual-node index
    vs = num_nodes
    existing = torch.arange(0, num_nodes, device=data.x.device)

    # Virtual → every atom (broadcast connection)
    e1 = torch.stack([torch.full((num_nodes,), vs, device=data.x.device), existing], dim=0)
    # Her atom → sanal
    e2 = torch.stack([existing, torch.full((num_nodes,), vs, device=data.x.device)], dim=0)

    data.edge_index = torch.cat([data.edge_index, e1, e2], dim=1)

    # Add edge attribute (bond_dim = current number of bond features)
    bond_dim = data.edge_attr.size(1)
    new_edge_attr = torch.zeros((num_nodes * 2, bond_dim), device=data.edge_attr.device)
    data.edge_attr = torch.cat([data.edge_attr, new_edge_attr], dim=0)

    return data



In [ ]:
# ----------------------- 3. DATASET -----------------------

class HIVDatasetVN(Dataset):
    """
    For each row:
    - SMILES → Mol → pyg_from_mol ile PyG Data objesi
    - data.global_desc: descriptor information as a [1×6] tensor
    - If virtual_node=True, add_virtual_node(data) is called
    """
    def __init__(self, df, augment=False, virtual_node=False):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.aug = augment
        self.virtual_node = virtual_node

    def len(self):
        return len(self.df)

    def get(self, idx):
        row = self.df.iloc[idx]
        smiles = row.smiles
        mol = mol_from_smiles(smiles, self.aug)
        data = pyg_from_mol(mol, row.HIV_active)

        # Add the descriptor as a 1×6 tensor
        desc = compute_descriptors(smiles)                                # shape (6,)
        data.global_desc = torch.tensor(desc, dtype=torch.float).unsqueeze(0)  # shape (1×6)

        # Batch index: 0 for every atom (single graph)
        data.batch = torch.tensor([0] * data.x.size(0), dtype=torch.long)

        if self.virtual_node:
            data = add_virtual_node(data)

        return data



In [ ]:
# ----------------------- 4. FOCAL LOSS -----------------------

from torchvision.ops import sigmoid_focal_loss

def focal_loss(inputs, targets, alpha=FCL_ALPHA, gamma=FCL_GAMMA):
    """
    Sigmoid Focal Loss uygular.
    - inputs: logits (batch_size)
    - targets: 0/1 labels (batch_size)
    """
    return sigmoid_focal_loss(inputs, targets, alpha=alpha, gamma=gamma, reduction='mean')

class GIN_MPNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS,
                 dropout=DROPOUT_RATE, desc_dim=0):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        # First GINConv layer (in_dim → hidden_dim)
        mlp1 = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.convs.append(GINConv(mlp1))
        self.bns.append(nn.BatchNorm1d(hidden_dim))

        # Remaining GIN layers (hidden_dim → hidden_dim)
        for _ in range(n_layers - 1):
            mlpi = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINConv(mlpi))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.read = global_mean_pool

        # Descriptor projection block (desc_dim → hidden_dim//4)
        if desc_dim > 0:
            self.desc_proj = nn.Sequential(
                nn.Linear(desc_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, hidden_dim // 4),
                nn.ReLU()
            )
            total_emb_dim = hidden_dim + hidden_dim // 4
        else:
            self.desc_proj = None
            total_emb_dim = hidden_dim

        # Son MLP: total_emb_dim → total_emb_dim//2 → total_emb_dim//4 → 1
        self.mlp = nn.Sequential(
            nn.Linear(total_emb_dim, total_emb_dim // 2),
            nn.BatchNorm1d(total_emb_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(total_emb_dim // 2, total_emb_dim // 4),
            nn.BatchNorm1d(total_emb_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(total_emb_dim // 4, 1)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        h = x
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)
            h = bn(h)
            h = F.relu(h)
            h = self.dropout(h)

        # Global mean pooling → graph-level vector (batch_size×hidden_dim)
        g = self.read(h, batch)

        # Concatenate descriptor projection if present
        if self.desc_proj is not None:
            # data.global_desc: shape (batch_size × desc_dim)
            d = self.desc_proj(data.global_desc.to(DEVICE))
            g = torch.cat([g, d], dim=1)

        out = self.mlp(g).view(-1)
        return out



In [ ]:
# ----------------------- 6. TRAINING AND VALIDATION FUNCTIONS -----------------------

def run_epoch_onecycle(loader, model, crit, optimizer, scheduler):
    """
    One training epoch with the OneCycleLR protocol, calling scheduler.step() after each batch.
    """
    model.train()
    total_loss, true_list, pred_list = 0.0, [], []

    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch)
        loss = crit(logits, batch.y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * batch.num_graphs
        true_list.append(batch.y.detach().cpu().numpy())
        pred_list.append(torch.sigmoid(logits.detach()).cpu().numpy())

    true_all = np.concatenate(true_list)
    pred_all = np.concatenate(pred_list)
    auc_score = roc_auc_score(true_all, pred_all)
    return total_loss / len(loader.dataset), auc_score

def run_validation(loader, model, crit, threshold=0.5):
    """
    Validation or test stage:
    Returns loss + Accuracy + Precision + Recall + F1 + ROC-AUC.
    """
    model.eval()
    total_loss, true_list, pred_list = 0.0, [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            loss = crit(logits, batch.y)

            total_loss += loss.item() * batch.num_graphs
            true_list.append(batch.y.detach().cpu().numpy())
            pred_list.append(torch.sigmoid(logits).detach().cpu().numpy())

    true_all = np.concatenate(true_list)
    pred_all = np.concatenate(pred_list)
    pred_bin = (pred_all >= threshold).astype(int)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(true_all, pred_bin),
        "precision": precision_score(true_all, pred_bin, zero_division=0),
        "recall": recall_score(true_all, pred_bin, zero_division=0),
        "f1": f1_score(true_all, pred_bin, zero_division=0),
        "roc_auc": roc_auc_score(true_all, pred_all)
    }

    return metrics, true_all, pred_all

def train_onecycle(train_loader, val_loader, sample_data, desc_dim):
    """
    Trains the model with an OneCycleLR scheduler and runs validation at the end of each epoch.
    """
    model = GIN_MPNN(
        in_dim=sample_data.x.size(1),
        hidden_dim=HIDDEN_DIM,
        n_layers=N_LAYERS,
        dropout=DROPOUT_RATE,
        desc_dim=desc_dim
    ).to(DEVICE)

    crit = focal_loss
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=WEIGHT_DECAY)

    total_steps = EPOCHS * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LR_BASE,
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy='cos',
        final_div_factor=10
    )

    best_val_auc = 0.0
    wait = 0
    history = {'tr_auc': [], 'va_auc': [], 'tr_ls': [], 'va_ls': []}

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_auc = run_epoch_onecycle(train_loader, model, crit, optimizer, scheduler)
        val_metrics, _, _ = run_validation(val_loader, model, crit)

        va_loss = val_metrics["loss"]
        va_auc = val_metrics["roc_auc"]

        history['tr_ls'].append(tr_loss)
        history['va_ls'].append(va_loss)
        history['tr_auc'].append(tr_auc)
        history['va_auc'].append(va_auc)

        print(
            f"Epoch {epoch:02d}  "
            f"Loss: {tr_loss:.4f}/{va_loss:.4f}  "
            f"AUC: {tr_auc:.3f}/{va_auc:.3f}  "
            f"Gap: {tr_auc - va_auc:.3f}"
        )

        if va_auc > best_val_auc + 1e-4:
            best_val_auc = va_auc
            wait = 0
            torch.save(model.state_dict(), "best_model_onecycle.pt")
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping tetiklendi.")
                break

    model.load_state_dict(torch.load("best_model_onecycle.pt"))
    return model, history, best_val_auc

In [ ]:
# ----------------------- 7. CROSS-VALIDATION AND TRAINING -----------------------

def cross_validate_training(df, k_folds=5):
    """
    Stratified K-Fold cross-validation:
    - Performs a train/val/test split for each fold.
    - Produces fold-wise Accuracy, Precision, Recall, F1, and ROC-AUC.
    - Mean ± std ve variance hesaplar.
    """
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=SEED)

    all_histories = []
    fold_results = []

    for fold, (train_idx, temp_idx) in enumerate(
            skf.split(df.smiles, df.HIV_active), start=1):

        print(f"\n{'='*25} FOLD {fold} {'='*25}")
        set_seed(SEED + fold)

        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_temp  = df.iloc[temp_idx].reset_index(drop=True)

        df_temp = df_temp.sample(frac=1.0, random_state=SEED + fold).reset_index(drop=True)
        half = len(df_temp) // 2
        df_val = df_temp[:half].reset_index(drop=True)
        df_test = df_temp[half:].reset_index(drop=True)

        print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

        train_ds = HIVDatasetVN(df_train, augment=True, virtual_node=True)
        val_ds   = HIVDatasetVN(df_val, augment=False, virtual_node=True)
        test_ds  = HIVDatasetVN(df_test, augment=False, virtual_node=True)

        labels_train = df_train.HIV_active.values
        class_counts = np.array([len(np.where(labels_train == t)[0]) for t in np.unique(labels_train)])
        weights = 1.0 / class_counts
        samples_weight = np.array([weights[int(t)] for t in labels_train])
        samples_weight = torch.from_numpy(samples_weight).double()
        sampler = WeightedRandomSampler(samples_weight, num_samples=len(samples_weight), replacement=True)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
        val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        sample_data = train_ds.get(0)
        model, history, best_val_auc = train_onecycle(
            train_loader, val_loader, sample_data, desc_dim=6
        )

        all_histories.append(history)

        crit = focal_loss
        val_metrics, val_true, val_pred = run_validation(val_loader, model, crit)
        test_metrics, test_true, test_pred = run_validation(test_loader, model, crit)

        fold_result = {
            "fold": fold,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1": test_metrics["f1"],
            "test_roc_auc": test_metrics["roc_auc"]
        }
        fold_results.append(fold_result)

        print(f"\nFold {fold} Validation Results:")
        print({k: round(v, 4) for k, v in fold_result.items() if k.startswith('val_')})

        print(f"Fold {fold} Test Results:")
        print({k: round(v, 4) for k, v in fold_result.items() if k.startswith('test_')})

    results_df = pd.DataFrame(fold_results)

    summary = {}
    test_metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]

    for col in test_metric_cols:
        summary[col] = {
            "mean": results_df[col].mean(),
            "std": results_df[col].std(ddof=1),
            "var": results_df[col].var(ddof=1)
        }

    print("\n" + "="*60)
    print("5-FOLD CROSS-VALIDATION TEST RESULTS")
    print("="*60)
    for col in test_metric_cols:
        print(
            f"{col}: "
            f"{summary[col]['mean']:.4f} ± {summary[col]['std']:.4f} "
            f"(var={summary[col]['var']:.6f})"
        )

    return all_histories, results_df, summary


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def pad_to_length(lst, target_len):
    if len(lst) >= target_len:
        return lst[:target_len]
    return lst + [lst[-1]] * (target_len - len(lst))

def plot_avg_training_history(histories, max_epoch=35):
    """
    histories: all_histories list. Each element is a history dictionary for one fold.
    max_epoch: number of epochs to use for each fold
    """
    tr_auc_matrix = []
    va_auc_matrix = []
    tr_ls_matrix  = []
    va_ls_matrix  = []

    for h in histories:
        tr_auc_matrix.append(pad_to_length(h['tr_auc'], max_epoch))
        va_auc_matrix.append(pad_to_length(h['va_auc'], max_epoch))
        tr_ls_matrix.append(pad_to_length(h['tr_ls'], max_epoch))
        va_ls_matrix.append(pad_to_length(h['va_ls'], max_epoch))

    # Mean calculations
    avg_tr_auc = np.mean(tr_auc_matrix, axis=0)
    avg_va_auc = np.mean(va_auc_matrix, axis=0)
    avg_tr_ls  = np.mean(tr_ls_matrix, axis=0)
    avg_va_ls  = np.mean(va_ls_matrix, axis=0)

    epochs = range(1, max_epoch + 1)

    # AUC plot
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, avg_tr_auc, label='Avg Train AUC', alpha=0.8, marker='o')
    plt.plot(epochs, avg_va_auc, label='Avg Val AUC', alpha=0.8, marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.title(f'Train vs. Validation AUC with Earlystopping')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # Loss plot
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, avg_tr_ls, label='Avg Train Loss', alpha=0.8, marker='o')
    plt.plot(epochs, avg_va_ls, label='Avg Val Loss', alpha=0.8, marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Train vs. Validation Loss with Earlystopping')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()



In [ ]:
# ----------------------- 9. ANA BLOK -----------------------
if __name__ == "__main__":
    df_all = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
    df_all = df_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    all_histories, results_df, summary = cross_validate_training(df_all, k_folds=5)

    print("\nFold-wise results:")
    display(results_df)

    results_df.to_csv("GDL_fold_results.csv", index=False)

    summary_rows = []
    for metric_name, vals in summary.items():
        summary_rows.append({
            "Metric": metric_name.replace("test_", "").upper(),
            "Mean": vals["mean"],
            "Std": vals["std"],
            "Variance": vals["var"],
            "Formatted": f"{vals['mean']:.4f} ± {vals['std']:.4f}"
        })

    summary_df = pd.DataFrame(summary_rows)
    print("\nSummary table:")
    display(summary_df)

    summary_df.to_csv("GDL_summary_results.csv", index=False)

In [ ]:
plot_avg_training_history(all_histories, max_epoch=35)

## GDL docking preparation block

This block is prepared for Colab and added for post-processing after prediction.

Added workflow:
- RDKit installation cell
- build a 13-column table from existing predictions/top scores
- top 10 candidates
- final 2 candidates
- colored 2D drawing
- `.smi` docking file

In [ ]:
# Colab RDKit install
import sys, subprocess, pkgutil

if pkgutil.find_loader("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

print("RDKit OK")

In [ ]:
# ==========================================
# GDL FINAL PIPELINE (MATCHED TO THIS NOTEBOOK)
# ==========================================

import numpy as np
import pandas as pd
import torch
from torch.utils.data import WeightedRandomSampler
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

# -------------------------------------------------
# 0) select the data source
# -------------------------------------------------
if "df_all" in globals():
    df_source = df_all.copy().reset_index(drop=True)
elif "df" in globals():
    df_source = df.copy().reset_index(drop=True)
else:
    raise ValueError("Ne df_all ne de df bulundu.")

if "results_df" not in globals():
    raise ValueError("results_df not found. Run the cross-validation cell first.")

if "cross_validate_training" not in globals():
    print("cross_validate_training is defined, but the final pipeline does not depend on it; continuing.")

# -------------------------------------------------
# 1) Select the best fold
# -------------------------------------------------
auc_col = "test_roc_auc" if "test_roc_auc" in results_df.columns else results_df.columns[-1]
best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
best_fold_number = best_fold_idx + 1

print(f"Using AUC column: {auc_col}")
print(f"Using best fold: {best_fold_number}")

# -------------------------------------------------
# 2) Rebuild the same split with the same logic as the notebook
# -------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (train_idx, temp_idx) in enumerate(
    skf.split(df_source.smiles, df_source.HIV_active), start=1
):
    if fold == best_fold_number:
        set_seed(SEED + fold)

        df_train = df_source.iloc[train_idx].reset_index(drop=True)
        df_temp  = df_source.iloc[temp_idx].reset_index(drop=True)

        df_temp = df_temp.sample(frac=1.0, random_state=SEED + fold).reset_index(drop=True)
        half = len(df_temp) // 2
        df_val = df_temp[:half].reset_index(drop=True)
        df_test = df_temp[half:].reset_index(drop=True)
        break

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

# -------------------------------------------------
# 3) Dataset / Loader
# -------------------------------------------------
train_ds = HIVDatasetVN(df_train, augment=True,  virtual_node=True)
val_ds   = HIVDatasetVN(df_val,   augment=False, virtual_node=True)
test_ds  = HIVDatasetVN(df_test,  augment=False, virtual_node=True)

labels_train = df_train.HIV_active.values
class_counts = np.array([len(np.where(labels_train == t)[0]) for t in np.unique(labels_train)])
weights = 1.0 / class_counts
samples_weight = np.array([weights[int(t)] for t in labels_train])
samples_weight = torch.from_numpy(samples_weight).double()
sampler = WeightedRandomSampler(samples_weight, num_samples=len(samples_weight), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# -------------------------------------------------
# 4) Retrain the model with the official training function from the notebook
# -------------------------------------------------
sample_data = train_ds.get(0)
model, history, best_val_auc = train_onecycle(
    train_loader, val_loader, sample_data, desc_dim=6
)

# -------------------------------------------------
# 5) Generate test predictions
# -------------------------------------------------
crit = focal_loss
test_metrics, test_true, test_pred = run_validation(test_loader, model, crit)

smiles_test = df_test["smiles"].reset_index(drop=True)
y_test = np.array(test_true).reshape(-1)
y_prob = np.array(test_pred).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# -------------------------------------------------
# 6) Prediction dataframe
# -------------------------------------------------
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# -------------------------------------------------
# 7) Top 10 candidates
# -------------------------------------------------
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(100).reset_index(drop=True)

# -------------------------------------------------
# 8) Descriptor hesaplama
# -------------------------------------------------
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# -------------------------------------------------
# 9) Shared 13 columns
# -------------------------------------------------
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGDL TOP 10 CANDIDATES:")
display(df_desc)

# -------------------------------------------------
# 10) Final 2 candidates
# -------------------------------------------------
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["y_prob", "QED"], ascending=False).head(2).copy()
else:
    print("Warning: No Lipinski-passing molecules found in top 100. Falling back to best available candidates.")
    final_df = df_desc.sort_values(by=["y_prob", "QED"], ascending=False).head(2).copy()
# -------------------------------------------------
# 11) Save
# -------------------------------------------------
df_desc.to_csv("GDL_top_10_candidates.csv", index=False)
final_df.to_csv("GDL_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("GDL_docking_input.smi", index=False, header=False)

print("\nSaved: GDL_top_10_candidates.csv")
print("Saved: GDL_final_2_candidates.csv")
print("Saved: GDL_docking_input.smi")

# -------------------------------------------------
# 12) Colored drawing
# -------------------------------------------------
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(350, 350),
    legends=[
        f"GDL Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"GDL Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# Top 2 molecules by QED
smiles_list = [
    "CC1=C(C(=O)Nc2ccc(Cl)c(C(=O)OC(C)C)c2)SCCO1",
    "CCN(CC)CC1CCCCN1CC(=O)N1c2ccccc2NC(=O)CC1C"
]

probs = [0.748, 0.734]  # your y_prob values (change if needed)
qeds  = [0.831, 0.817]

mols = [Chem.MolFromSmiles(sm) for sm in smiles_list]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(350, 350),
    legends=[
        f"Mol1\nQED={qeds[0]:.3f}\nProb={probs[0]:.3f}",
        f"Mol2\nQED={qeds[1]:.3f}\nProb={probs[1]:.3f}"
    ]
)

display(img)